# Building a Networking-Focused LLM: Complete Guide 

This notebook walks through the entire process of creating and training a language model specialized in networking concepts **using QLoRA (Quantized LoRA)** for memory-efficient training.

## What is QLoRA?

**QLoRA** combines:
- **Quantization**: Loads the base model in 4-bit precision (~75% memory reduction)
- **LoRA**: Trains small adapter layers instead of the full model

This allows training large models (like Mistral 7B) on limited hardware like Google Colab!

## What is a Networking-Focused LLM?

A **Large Language Model (LLM)** is a neural network trained to understand and generate human-like text. A **networking-focused LLM** is specifically fine-tuned on networking terminology, protocols, and concepts to:
- Answer networking questions
- Classify network configurations
- Generate network documentation
- Troubleshoot network issues

### When to Use Small Language Models (SLMs)?

For domain-specific tasks like networking, **Small Language Models** (typically <3B parameters) offer several advantages:
- **Faster training**: Takes minutes instead of hours
- **Lower cost**: Runs on consumer GPUs or even CPUs
- **Better focus**: Less general knowledge means more specialized performance
- **Easier deployment**: Can run on edge devices and local machines

## Table of Contents
1. [Setup & Data Preparation](#1.-data-preparation-for-llm)
2. [Model Selection](#2.-model-selection)
3. [Data Preprocessing & Tokenization](#3.-data-preprocessing--tokenization)
4. [Training Parameters](#4.-configure-training-parameters)
5. [Training & Monitoring](#5.-training-with-progress-monitoring)
6. [Evaluation & Testing](#6.-testing-and-evaluation)
7. [Model Testing](#7.-test-the-model)
8. [Saving Your Model](#8.-save-the-model)

---

**⚡ This QLoRA version is optimized for Google Colab and limited GPU memory!**


---


## Episode 1: Introduction to Fine-Tuning LLMs for Networking Use Cases

Basic concepts, why fine-tune, where to find models, and practical use cases for network operations. (Episode 1)

### 1.1 Installing QLoRA Requirements

**Run this cell first to install required packages:**


In [ ]:
# Install QLoRA requirements (bitsandbytes for 4-bit quantization)
# Note: Requires CUDA-enabled GPU. Won't work on CPU-only systems.

!pip install -q bitsandbytes accelerate

print("✓ QLoRA requirements installed successfully!")
print("  • bitsandbytes: For 4-bit quantization")
print("  • accelerate: For efficient model loading")
print("\n⚠️  Note: Make sure you have GPU enabled in Google Colab")
print("   Runtime → Change runtime type → GPU")


In [ ]:
#Libraries and packages

# Core PyTorch - Deep learning framework
import torch  # Main PyTorch library for tensor operations and neural networks
print ("done1")
# PyTorch Data Utilities
from torch.utils.data import (
    Dataset,        # Base class for creating custom datasets
    DataLoader,     # Handles batching, shuffling, and loading data efficiently
    random_split    # Splits dataset into train/validation/test sets
)

# Hugging Face Transformers - Pre-trained models and utilities
from transformers import (
    AutoTokenizer,              # Automatically loads the right tokenizer for any model
    AutoModelForCausalLM,       # Loads pre-trained causal language models (GPT-style)
    TrainingArguments,          # Configuration object for training settings
    Trainer,                    # High-level API that handles training loop
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig          # For 4-bit quantization (QLoRA)
)

# Data Processing
import pandas as pd  # DataFrame operations for structured data
import numpy as np   # Numerical operations and array manipulation
print ("done2")

# Evaluation Metrics
from sklearn.metrics import (
    confusion_matrix,      # Shows prediction accuracy across categories
    classification_report  # Detailed precision, recall, F1 scores
)

# Visualization
import seaborn as sns           # Statistical data visualization
import matplotlib.pyplot as plt # Plotting library
print ("done3")

# Progress Tracking
from tqdm.auto import tqdm  # Creates progress bars for loops

#
from huggingface_hub import login
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
import json

import re
import json
import hashlib
from typing import List, Dict, Any
import pandas as pd
from datetime import datetime
import random


print ("Successfully completed environment setup")
print ("✓ QLoRA-specific imports loaded (BitsAndBytesConfig, prepare_model_for_kbit_training)")


##1.2 Why Fine-Tune for Networking?

General models like GPT-4 hallucinate on networking-specific tasks because they weren't trained on enough networking data. (Episode 1)

In [ ]:
# Example of what happens with general models on networking tasks
sample_cisco_config = """
interface GigabitEthernet0/1
 description Link to Core Switch
 ip address 192.168.1.1 255.255.255.0
 no shutdown
"""

print("Sample Cisco Configuration:")
print(sample_cisco_config)
print("\nChallenges with general LLMs:")
print("1. May suggest non-existent commands")
print("2. Misinterprets SNMP traps")
print("3. Doesn't understand vendor-specific syntax")
print("4. Lacks context of your specific network topology")

---
### Tips for Model Selection

### Understanding Model Architecture

**Causal Language Models** (like GPT) predict the next word in a sequence. They're ideal for:
- Text generation
- Completion tasks
- Question answering

### Model Size Considerations

| Model | Parameters | Memory Required | Speed | Best For | Networking Score | Authentication |
|-------|-----------|-----------------|-------|----------|------------------|----------------|
| **DistilGPT-2** | 82M | ~330MB | Fastest | Quick prototyping, testing, CPU-friendly | 6/10 | ✅ None |
| **GPT-2** | 124M | ~500MB | Very Fast | General networking text generation | 7/10 | ✅ None |
| **TinyLlama** | 1.1B | ~2GB | Very Fast | Ultra-fast testing, limited resources | 6/10 | ✅ None |
| **Gemma 2B** | 2B | ~4GB | Fast | Efficient reasoning, networking analysis | 7/10 | 🔐 **Required** |
| **Phi-3 Mini** | 3.8B | ~8GB | Fast | Technical/reasoning tasks, troubleshooting | 8/10 | 🔐 May be required |
| **CodeLlama 7B** | 7B | ~14GB | Medium | Configuration generation, structured commands | 8/10 | 🔐 May be required |
| **Mistral 7B** | 7B | ~14GB | Medium | High-quality responses, complex configs (requires GPU) | 9/10 | ✅ Generally open |

**Note:** Memory requirements are for inference. Training requires approximately 2-3x more memory depending on batch size and optimization settings.

### 🔐 Model Authentication Requirements

Some models are **gated** and require Hugging Face authentication:

**Gemma 2B** (🔐 Required):
- Accept license at https://huggingface.co/google/gemma-2b
- Requires Hugging Face account and token
- Add to notebook: `from huggingface_hub import login; login(token="your_token")`

**Recommended: Start with freely accessible models** (DistilGPT-2, GPT-2, TinyLlama) to avoid authentication setup.


### Loading the Model

The `AutoModelForCausalLM` class automatically loads the correct model architecture.
The `AutoTokenizer` loads the corresponding tokenizer that converts text to tokens.

**What is a Tokenizer?**
- Converts text into numerical tokens (e.g., "BGP protocol" → [45, 892, 37])
- Each model has its own vocabulary and tokenization rules


## Episode 2: Data Preparation and Organization for Networking Use Cases

Structuring networking data into JSONL format, data types, and preparation procedures. (Episode 2)

### 2.1 Understanding JSONL Format

The most common format for fine-tuning datasets is JSON Lines (JSONL), where each line is a complete JSON object. (Episode 2)

In [ ]:
# Example JSONL structure for networking data
sample_training_data = [
    {
        "instruction": "Analyze this BGP configuration and identify any issues",
        "input": """
        router bgp 65001
         neighbor 192.168.1.2 remote-as 65002
         neighbor 192.168.1.2 description ISP_CONNECTION
         network 10.0.0.0 mask 255.0.0.0
        """,
        "output": "The BGP configuration looks correct. AS 65001 is peering with AS 65002 via neighbor 192.168.1.2. The network 10.0.0.0/8 is being advertised. No obvious issues detected."
    },
    {
        "instruction": "Diagnose this network issue from the log entries",
        "input": "2024-12-11 10:30:15: %LINK-3-UPDOWN: Interface GigabitEthernet0/1, changed state to down",
        "output": "Interface GigabitEthernet0/1 has gone down. This could indicate a physical layer issue, cable problem, or remote device failure. Check physical connections and interface statistics."
    },
    {
        "instruction": "Generate VLAN configuration for network segmentation",
        "input": "Need to create VLANs for: Sales (VLAN 10), Engineering (VLAN 20), Guest (VLAN 30)",
        "output": """
        vlan 10
         name Sales
        vlan 20
         name Engineering
        vlan 30
         name Guest
        """
    }
]

# Save as JSONL file
with open('networking_training_data.jsonl', 'w') as f:
    for item in sample_training_data:
        f.write(json.dumps(item) + '\n')

print("Created networking_training_data.jsonl with sample data")
print(f"Total training examples: {len(sample_training_data)}")

##2.2 Data Preparation

Data preparation is 80% of the effort when optimizing models. (Episode 2)

In [ ]:
def prepare_networking_data(raw_configs=None, raw_logs=None, raw_tickets=None, raw_metrics=None, use_existing_sample=True):
    """
    Comprehensive data preparation pipeline for networking data
    Actually performs the steps covered in Episode 2:
    1. Collection
    2. Cleaning and Normalization
    3. Chunking
    4. Anonymization
    5. Annotation (creating instruction-response pairs)

    Args:
        use_existing_sample: If True, uses the sample_training_data from Episode 2
        raw_*: Actual raw data to process (optional)
    """

    prepared_data = []

    print("EPISODE 2: DATA PREPARATION - LIVE IMPLEMENTATION")
    print("=" * 60)

    # ===== STEP 1: COLLECTION =====
    print("\nStep 1: Collection - Gathering data from various sources")

    collected_configs = []
    collected_logs = []
    collected_tickets = []
    collected_metrics = []

    # Option 1: Use existing sample data from Episode 2
    if use_existing_sample and 'sample_training_data' in globals():
        print("✅ Using existing sample_training_data from Episode 2")
        print(f"✅ Found {len(sample_training_data)} existing training examples")

        # Extract raw data from the existing structured samples
        for item in sample_training_data:
            input_text = item.get('input', '')

            # Identify what type of raw data this represents
            if any(keyword in input_text.lower() for keyword in ['interface', 'router bgp', 'vlan', 'switchport']):
                collected_configs.append(input_text)
                print(f"   Extracted config: {input_text[:50]}...")

            elif '%' in input_text and any(keyword in input_text for keyword in ['UPDOWN', 'LINK', 'BGP']):
                collected_logs.append(input_text)
                print(f"   Extracted log: {input_text[:50]}...")

            elif item['instruction'].lower().startswith('generate'):
                # This was a generation request, treat the output as raw config
                collected_configs.append(item['output'])
                print(f"   Extracted generated config: {item['output'][:50]}...")

    # Option 2: Use provided raw data
    if raw_configs:
        collected_configs.extend(raw_configs)
        print(f"✅ Added {len(raw_configs)} provided configurations")

    if raw_logs:
        collected_logs.extend(raw_logs)
        print(f"✅ Added {len(raw_logs)} provided log entries")

    if raw_tickets:
        collected_tickets.extend(raw_tickets)
        print(f"✅ Added {len(raw_tickets)} provided tickets")

    if raw_metrics:
        collected_metrics.extend(raw_metrics)
        print(f"✅ Added {len(raw_metrics)} provided metrics")

    # Show what we actually collected
    print(f"\nCOLLECTION SUMMARY (what we're actually working with):")
    print(f"Configurations: {len(collected_configs)}")
    print(f"Log entries: {len(collected_logs)}")
    print(f"Trouble tickets: {len(collected_tickets)}")
    print(f"Metric data points: {len(collected_metrics)}")

    # If we have no data at all, explain this
    if not any([collected_configs, collected_logs, collected_tickets, collected_metrics]):
        print("\n⚠️  No raw data found to process!")
        print("   This would typically happen when:")
        print("   1. sample_training_data doesn't exist")
        print("   2. No raw data was provided as parameters")
        print("   3. You want to see the process with sample data")
        print("\nTo demonstrate the process, set add_demo_data=True")
        return []

    # ===== STEP 2: CLEANING AND NORMALIZATION =====
    print(f"\nStep 2: Cleaning and Normalization")
    print("- Standardizing formats")
    print("- Removing irrelevant information")
    print("- Redacting sensitive data (PII, confidential operational details)")
    print("- Parsing structured data into consistent text representation")

    def anonymize_ip_addresses(text):
        """Replace real IP addresses with anonymized ones"""
        ip_mapping = {}
        ip_pattern = r'\b(?:\d{1,3}\.){3}\d{1,3}\b'

        def replace_ip(match):
            real_ip = match.group(0)
            if real_ip not in ip_mapping:
                hash_obj = hashlib.md5(real_ip.encode())
                hash_hex = hash_obj.hexdigest()
                octets = [str(int(hash_hex[i:i+2], 16) % 254 + 1) for i in range(0, 8, 2)]
                ip_mapping[real_ip] = f"10.{octets[1]}.{octets[2]}.{octets[3]}"
            return ip_mapping[real_ip]

        return re.sub(ip_pattern, replace_ip, text), ip_mapping

    def clean_config(config):
        """Clean and normalize configuration data"""
        if not config.strip():
            return "", {}

        # Remove empty lines and comments
        lines = [line.strip() for line in config.split('\n') if line.strip() and not line.strip().startswith('!')]

        clean_config = '\n'.join(lines)
        clean_config, ip_map = anonymize_ip_addresses(clean_config)

        return clean_config, {'ip_mapping': ip_map}

    # Clean only the data we actually have
    cleaned_configs = []
    if collected_configs:
        print(f"Cleaning {len(collected_configs)} configurations...")
        for i, config in enumerate(collected_configs):
            clean_cfg, mapping = clean_config(config)
            if clean_cfg:  # Only add non-empty configs
                cleaned_configs.append(clean_cfg)
                print(f"   Config {i+1}: {len(clean_cfg)} characters after cleaning")

    cleaned_logs = []
    if collected_logs:
        print(f"Cleaning {len(collected_logs)} log entries...")
        for i, log in enumerate(collected_logs):
            clean_log, mapping = anonymize_ip_addresses(log)
            cleaned_logs.append(clean_log)
            print(f"   Log {i+1}: {log[:50]}... → {clean_log[:50]}...")

    print("✅ Cleaning completed")

    # ===== STEP 3: CHUNKING =====
    print(f"\nStep 3: Chunking - Breaking down large documents into manageable segments")

    def chunk_config(config, max_lines=10):
        if not config.strip():
            return []

        lines = config.split('\n')
        chunks = []
        current_chunk = []
        current_context = ""

        for line in lines:
            if line.startswith(('interface ', 'router ', 'hostname ')):
                if current_chunk:
                    chunks.append({
                        'context': current_context,
                        'content': '\n'.join(current_chunk)
                    })
                current_chunk = [line]
                current_context = line.strip()
            else:
                current_chunk.append(line)

            if len(current_chunk) >= max_lines:
                chunks.append({
                    'context': current_context,
                    'content': '\n'.join(current_chunk)
                })
                current_chunk = []
                current_context = ""

        if current_chunk:
            chunks.append({
                'context': current_context,
                'content': '\n'.join(current_chunk)
            })

        return chunks

    config_chunks = []
    for i, config in enumerate(cleaned_configs):
        chunks = chunk_config(config)
        for j, chunk in enumerate(chunks):
            config_chunks.append({
                'source': f'config_{i}_chunk_{j}',
                'type': 'configuration',
                'context': chunk['context'],
                'content': chunk['content']
            })

    print(f"✅ Created {len(config_chunks)} configuration chunks from {len(cleaned_configs)} configs")

    # ===== STEP 4: ANNOTATION - CREATE INSTRUCTION-RESPONSE PAIRS =====
    print(f"\nStep 4: Creating instruction-response pairs from our actual data")

    # Process configurations we actually have
    for chunk in config_chunks:
        instruction = "Analyze this network configuration and identify any potential issues or improvements."
        input_text = chunk['content']

        # Generate context-appropriate responses
        if 'interface' in input_text.lower():
            if 'trunk' in input_text.lower():
                output = "This is a trunk interface configuration. The interface allows VLANs as specified. Configuration appears standard for inter-switch connectivity."
            elif 'access' in input_text.lower():
                output = "This is an access port configuration assigned to a specific VLAN. Configuration appears appropriate for end-device connectivity."
            else:
                output = "This interface configuration should specify whether it's an access or trunk port for clarity."
        elif 'router bgp' in input_text.lower():
            output = "BGP configuration detected. Neighbor relationship is configured with remote AS. Ensure proper route filtering and security policies are in place."
        elif 'vlan' in input_text.lower():
            output = "VLAN configuration found. This appears to be creating VLANs for network segmentation."
        else:
            output = "Configuration appears standard. Review against your organization's standards."

        prepared_data.append({
            "instruction": instruction,
            "input": input_text,
            "output": output,
            "source": chunk['source'],
            "type": "config_analysis"
        })

    # Process logs we actually have
    for i, log in enumerate(cleaned_logs):
        instruction = "Analyze this network log entry and provide troubleshooting guidance."
        input_text = log

        if "changed state to down" in log.lower():
            output = "Interface has gone down. Check physical connections, cable integrity, and remote device status. Review interface error counters for additional clues."
        elif "bgp" in log.lower() and "down" in log.lower():
            output = "BGP neighbor session has gone down. Verify network connectivity to the peer and check for recent configuration changes."
        else:
            output = "Log entry indicates a network event. Monitor for patterns and correlate with other events if this represents an ongoing issue."

        prepared_data.append({
            "instruction": instruction,
            "input": input_text,
            "output": output,
            "source": f"log_{i}",
            "type": "log_analysis"
        })

    # ===== FINAL STATISTICS =====
    print(f"\nFINAL RESULTS:")
    print(f"Processed data from:")
    if use_existing_sample and 'sample_training_data' in globals():
        print(f"  - Episode 2 sample_training_data: {len(sample_training_data)} original examples")
    if collected_configs:
        print(f"  - {len(collected_configs)} configuration files")
    if collected_logs:
        print(f"  - {len(collected_logs)} log entries")
    if collected_tickets:
        print(f"  - {len(collected_tickets)} trouble tickets")
    if collected_metrics:
        print(f"  - {len(collected_metrics)} metric data points")

    print(f"\nGenerated {len(prepared_data)} new instruction-response pairs:")
    type_counts = {}
    for item in prepared_data:
        item_type = item['type']
        type_counts[item_type] = type_counts.get(item_type, 0) + 1

    for item_type, count in type_counts.items():
        print(f"  - {item_type}: {count} examples")

    # Save the results
    output_file = 'processed_networking_data.jsonl'
    with open(output_file, 'w') as f:
        for item in prepared_data:
            f.write(json.dumps(item) + '\n')

    print(f"\n✅ Saved processed data to {output_file}")

    return prepared_data

# Now let's run it and see what we actually get from our Episode 2 data
print("PROCESSING ONLY THE DATA FROM EPISODE 2:")
print("=" * 50)

processed_data = prepare_networking_data(use_existing_sample=True)

print(f"\nSUMMARY:")
print(f"Original Episode 2 data: {len(sample_training_data)} examples")
print(f"New processed data: {len(processed_data)} examples")
print(f"No additional data was automatically added!")

In [ ]:
# If you want to see the full process with more sample data, call it explicitly:
def add_demo_data():
    """Add comprehensive demo data to see the full data preparation process"""

    demo_configs = [
        """
        hostname CORE-SW-01
        !
        interface GigabitEthernet0/1
         description Link to firewall 192.168.1.100
         ip address 10.0.1.1 255.255.255.0
         switchport mode trunk
         switchport trunk allowed vlan 10,20,30,99
         no shutdown
        """
    ]

    demo_logs = [
        "2024-12-11 10:30:15 CORE-SW-01: %LINK-3-UPDOWN: Interface GigabitEthernet0/1, changed state to down"
    ]

    return prepare_networking_data(
        raw_configs=demo_configs,
        raw_logs=demo_logs,
        use_existing_sample=True
    )

# Uncomment this if you want to see the process with additional demo data:
demo_processed_data = add_demo_data()

#### Exercise Tips
1. Replace this with your own data (CSV, JSON, database)
2. Expand to hundreds or thousands of examples
3. Include diverse networking topics
4. Ensure consistent labeling

###Episode 3 - Selecting an LLM

Different tiers of model access, evaluation platforms, and selection criteria.

####3.1 Model Selection Tiers



Three tiers of model access based on technical comfort level and needs.

In [ ]:
# Tier 1: No-code playgrounds (Episode 3)
print("TIER 1: No-code Playgrounds")
print("- LM Studio (GUI interface)")
print("- Hugging Face Spaces")
print("- Perfect for initial exploration")
print()

# Tier 2: Local execution (Episode 3)
print("TIER 2: Local Execution")
print("- LM Studio for local GUI")
print("- Ollama for command-line interface")
print("- Complete control and security")
print()

# Tier 3: API access (Episode 3)
print("TIER 3: Free API Access")
print("- Groq (fast inference)")
print("- GitHub Models")
print("- Perfect for prototyping")

###3.2 Hardware Requirements

 Understanding compute requirements for different model sizes.

In [ ]:
def check_hardware_requirements():
    """
    Hardware requirements covered in Episode 3
    """
    requirements = {
        "Small Models (3B-7B)": {
            "RAM": "8-16GB",
            "GPU": "Optional, RTX 3060+ recommended",
            "Examples": "Llama 3.2 (3B), Phi-3-mini"
        },
        "Medium Models (7B-13B)": {
            "RAM": "16-32GB",
            "GPU": "RTX 4090 (24GB VRAM) or similar",
            "Examples": "Llama 3.1 (8B), Mistral 7B"
        },
        "Large Models (70B+)": {
            "RAM": "64GB+",
            "GPU": "Multiple GPUs or 80GB+ VRAM",
            "Examples": "Llama 3.1 (70B)"
        }
    }

    for size, specs in requirements.items():
        print(f"{size}:")
        for spec, value in specs.items():
            print(f"  {spec}: {value}")
        print()

check_hardware_requirements()

### 3.3 Model Selection for GPT - 5

Since we're using the latest version of GPT-5, let's configure it properly.

In [ ]:
def check_hardware_requirements():
    """
    Hardware requirements covered in Episode 3
    """
    requirements = {
        "Small Models (3B-7B)": {
            "RAM": "8-16GB",
            "GPU": "Optional, RTX 3060+ recommended",
            "Examples": "Llama 3.2 (3B), Phi-3-mini"
        },
        "Medium Models (7B-13B)": {
            "RAM": "16-32GB",
            "GPU": "RTX 4090 (24GB VRAM) or similar",
            "Examples": "Llama 3.1 (8B), Mistral 7B"
        },
        "Large Models (70B+)": {
            "RAM": "64GB+",
            "GPU": "Multiple GPUs or 80GB+ VRAM",
            "Examples": "Llama 3.1 (70B)"
        }
    }

    for size, specs in requirements.items():
        print(f"{size}:")
        for spec, value in specs.items():
            print(f"  {spec}: {value}")
        print()

check_hardware_requirements()

In [ ]:
# ============================================================================
# ACTUAL MODEL LOADING WITH 4-BIT QUANTIZATION (QLoRA)
# ============================================================================

print("="*80)
print("MODEL SELECTION FOR NETWORKING TASKS (QLoRA)")
print("="*80)

# Available models optimized for different use cases
NETWORKING_MODELS = {
    'distilgpt2': {
        'model_id': 'distilgpt2',
        'params': '82M',
        'memory': '~330MB',
        'memory_4bit': '~100MB',
        'speed': 'Fastest',
        'best_for': 'Quick prototyping, testing, CPU-friendly',
        'networking_score': 6,
        'auth_required': False
    },
    'gpt2': {
        'model_id': 'gpt2',
        'params': '124M',
        'memory': '~500MB',
        'memory_4bit': '~150MB',
        'speed': 'Very Fast',
        'best_for': 'General networking text generation',
        'networking_score': 7,
        'auth_required': False
    },
    'tinyllama': {
        'model_id': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
        'params': '1.1B',
        'memory': '~2GB',
        'memory_4bit': '~600MB',
        'speed': 'Very Fast',
        'best_for': 'Ultra-fast testing, limited resources',
        'networking_score': 6,
        'auth_required': False
    },
    'gemma-2b': {
        'model_id': 'google/gemma-2b',
        'params': '2B',
        'memory': '~4GB',
        'memory_4bit': '~1.2GB',
        'speed': 'Fast',
        'best_for': 'Efficient reasoning, networking analysis',
        'networking_score': 7,
        'auth_required': True  # 🔐 Requires HuggingFace token
    },
    'phi-3-mini': {
        'model_id': 'microsoft/Phi-3-mini-4k-instruct',
        'params': '3.8B',
        'memory': '~8GB',
        'memory_4bit': '~2.5GB',
        'speed': 'Fast',
        'best_for': 'Technical/reasoning tasks, good for troubleshooting',
        'networking_score': 8,
        'auth_required': True  # 🔐 May require license agreement
    },
    'codellama-7b': {
        'model_id': 'codellama/CodeLlama-7b-hf',
        'params': '7B',
        'memory': '~14GB',
        'memory_4bit': '~3.5GB',
        'speed': 'Medium',
        'best_for': 'Configuration generation, structured commands',
        'networking_score': 8,
        'auth_required': True  # 🔐 May require Meta license
    },
    'mistral-7b': {
        'model_id': 'mistralai/Mistral-7B-v0.1',
        'params': '7B',
        'memory': '~14GB',
        'memory_4bit': '~3.5GB',
        'speed': 'Medium',
        'best_for': 'High-quality responses, complex configs (requires GPU)',
        'networking_score': 9,
        'auth_required': False  # Generally open access
    }
}

print("\nAvailable Models for Networking (with 4-bit quantization):\n")
print(f"{'Model':<15} {'Size':<8} {'4-bit RAM':<12} {'Speed':<12} {'Score':<8} {'Auth':<8} {'Best For'}")
print("-" * 120)

for name, info in NETWORKING_MODELS.items():
    score_visual = "★" * info['networking_score'] + "☆" * (10 - info['networking_score'])
    auth_status = "🔐 Yes" if info['auth_required'] else "✅ No"
    print(f"{name:<15} {info['params']:<8} {info['memory_4bit']:<12} {info['speed']:<12} {score_visual:<8} {auth_status:<8} {info['best_for']}")

print("\n" + "="*80)

# SELECT YOUR MODEL HERE
# Options: 'distilgpt2', 'gpt2', 'tinyllama', 'gemma-2b', 'phi-3-mini', 'codellama-7b', 'mistral-7b'

SELECTED_MODEL = 'mistral-7b'  # ← Using Mistral 7B with QLoRA for Google Colab

print(f"\n🎯 SELECTED: {SELECTED_MODEL}")
print(f"   {NETWORKING_MODELS[SELECTED_MODEL]['best_for']}")
print(f"   Parameters: {NETWORKING_MODELS[SELECTED_MODEL]['params']}")
print(f"   Memory (full precision): {NETWORKING_MODELS[SELECTED_MODEL]['memory']}")
print(f"   Memory (4-bit QLoRA): {NETWORKING_MODELS[SELECTED_MODEL]['memory_4bit']} ⚡")
print(f"   Networking Score: {NETWORKING_MODELS[SELECTED_MODEL]['networking_score']}/10")

# Check authentication requirement
if NETWORKING_MODELS[SELECTED_MODEL]['auth_required']:
    print(f"\n🔐 AUTHENTICATION REQUIRED for {SELECTED_MODEL}!")
    print(f"   This model requires Hugging Face authentication.")
    print(f"   Steps to authenticate:")
    print(f"   1. Create account at https://huggingface.co/join")
    print(f"   2. Accept model license at https://huggingface.co/{NETWORKING_MODELS[SELECTED_MODEL]['model_id']}")
    print(f"   3. Get your token from https://huggingface.co/settings/tokens")
    print(f"   4. Run: from huggingface_hub import login; login(token='your_token_here')")
    print(f"\n   💡 TIP: To skip authentication, use 'distilgpt2', 'gpt2', or 'tinyllama'")

# Set up device
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n📍 Device: {device}")

if device.type == 'cuda':
    print(f"   ✓ GPU detected - QLoRA will work efficiently!")
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print(f"   ⚠️  Warning: No GPU detected. QLoRA works best with GPU.")
    print(f"   💡 In Google Colab: Runtime > Change runtime type > GPU")

# ============================================================================
# CONFIGURE 4-BIT QUANTIZATION (QLoRA)
# ============================================================================
print("\n" + "="*80)
print("CONFIGURING 4-BIT QUANTIZATION")
print("="*80)

# QLoRA quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # Enable 4-bit loading
    bnb_4bit_quant_type="nf4",              # Normal Float 4-bit (optimal for neural networks)
    bnb_4bit_compute_dtype=torch.float16,  # Compute in float16 for speed
    bnb_4bit_use_double_quant=True,        # Double quantization for extra compression
)

print("✓ 4-bit Quantization Configuration:")
print(f"  • Quantization type: NF4 (Normal Float 4-bit)")
print(f"  • Compute dtype: float16")
print(f"  • Double quantization: Enabled")
print(f"  • Expected memory savings: ~75%")

# Load model and tokenizer
print(f"\n🔄 Loading {NETWORKING_MODELS[SELECTED_MODEL]['model_id']} in 4-bit...")

from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = NETWORKING_MODELS[SELECTED_MODEL]['model_id']

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,  # Apply 4-bit quantization
    device_map="auto",               # Automatically distribute across available devices
    trust_remote_code=True,
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

print(f"\n✓ Loaded {SELECTED_MODEL} with 4-bit quantization!")
print(f"  Model ID: {model_id}")
print(f"  Memory footprint reduced by ~75%")
print(f"  Model ready for QLoRA training!")

print("\n" + "="*80)
print("💡 NEXT: The model will be wrapped with LoRA adapters in the training cell")
print("="*80)


##Episode 4: Pre-Training Setup and Configuration

Fine-tuning approaches (LoRA vs full fine-tuning), environment setup, and hyperparameter configuration.

###4.1 Fine Tuning Approaches

Parameter-Efficient Fine-Tuning (PEFT) methods vs full fine-tuning.

In [ ]:
print("FINE-TUNING APPROACHES (Episode 4):")
print()
print("FULL FINE-TUNING:")
print("- Updates all 7+ billion parameters")
print("- Like replacing every router in your network")
print("- Expensive, time-consuming, risky")
print("- Rarely needed for domain-specific tasks")
print()

print("PARAMETER-EFFICIENT FINE-TUNING (PEFT):")
print("- Updates <1% of parameters")
print("- Like adding route maps/ACLs to existing infrastructure")
print("- Fast, cost-effective, lower risk")
print("- Recommended for networking use cases")
print()

peft_methods = {
    "LoRA (Low-Rank Adaptation)": {
        "description": "Adds small trainable matrices alongside frozen base model",
        "pros": "Fast training, low memory, swappable adapters",
        "cons": "Slightly lower performance than full fine-tuning",
        "use_case": "Most networking applications"
    },
    "QLoRA": {
        "description": "LoRA + quantized base model",
        "pros": "Even lower memory usage, can train larger models",
        "cons": "Slightly longer training time",
        "use_case": "Limited GPU memory scenarios"
    },
    "Adapter Layers": {
        "description": "Inserts small neural network layers between existing layers",
        "pros": "Easy to swap in/out",
        "cons": "Small inference overhead",
        "use_case": "Multiple specialized tasks"
    },
    "Prefix Tuning": {
        "description": "Trains special prompt tokens",
        "pros": "Extremely parameter-efficient (0.1%)",
        "cons": "Typically lower performance",
        "use_case": "Severe memory constraints"
    }
}

for method, details in peft_methods.items():
    print(f"{method}:")
    for key, value in details.items():
        print(f"  {key}: {value}")
    print()

###4.2 Environment Setup

Installing required packages and configuring training environment. (Episode 4)

In [ ]:
# Training configuration for LoRA fine-tuning
from dataclasses import dataclass
from typing import Optional

@dataclass
class NetworkingTrainingConfig:
    """
    Training configuration based on Episode 4 recommendations
    """
    # Model settings
    model_name: str = "microsoft/DialoGPT-medium"  # Smaller model for demo

    # LoRA settings (Episode 4 defaults)
    lora_r: int = 16  # Rank
    lora_alpha: int = 32  # Usually 2x rank
    lora_dropout: float = 0.05

    # Training hyperparameters (Episode 4 recommendations)
    learning_rate: float = 2e-4  # 0.0002
    batch_size: int = 4
    gradient_accumulation_steps: int = 4  # Effective batch size = 16
    num_epochs: int = 3
    warmup_steps: int = 100

    # Environment settings
    output_dir: str = "./outputs/networking-model-v1"
    logging_steps: int = 50
    save_steps: int = 100
    evaluation_strategy: str = "steps"
    eval_steps: int = 100

# Initialize configuration
config = NetworkingTrainingConfig()

print("TRAINING CONFIGURATION (Episode 4):")
print(f"Learning Rate: {config.learning_rate} (small for fine-tuning)")
print(f"Batch Size: {config.batch_size}")
print(f"Gradient Accumulation: {config.gradient_accumulation_steps}")
print(f"Effective Batch Size: {config.batch_size * config.gradient_accumulation_steps}")
print(f"Epochs: {config.num_epochs}")
print()
print("LoRA Settings:")
print(f"  Rank (r): {config.lora_r}")
print(f"  Alpha: {config.lora_alpha}")
print(f"  Dropout: {config.lora_dropout}")

4.3 Data Loading and Preprocessing

In [ ]:
def load_and_preprocess_data(jsonl_file):
    """
    Load JSONL data and preprocess for training
    Based on Episode 2 data preparation
    """
    data = []

    try:
        with open(jsonl_file, 'r') as f:
            for line in f:
                data.append(json.loads(line.strip()))
    except FileNotFoundError:
        print(f"File {jsonl_file} not found. Using sample data.")
        data = sample_training_data  # From Episode 2

    # Convert to training format
    formatted_data = []
    for item in data:
        # Create instruction-following format
        text = f"Instruction: {item['instruction']}\n"
        if 'input' in item and item['input']:
            text += f"Input: {item['input']}\n"
        text += f"Response: {item['output']}"

        formatted_data.append({"text": text})

    return formatted_data

# Load and prepare data
training_data = load_and_preprocess_data('networking_training_data.jsonl')
print(f"Loaded {len(training_data)} training examples")
print("\nSample formatted data:")
print(training_data[0]["text"][:200] + "...")

##Episode 5: Training and Monitoring

Understanding loss curves, monitoring training progress, and avoiding overfitting.

###5.1 Setting Up Training with Monitoring

Understanding training metrics and monitoring progress.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

def setup_training_monitoring():
    """
    Set up training with proper monitoring
    Episode 5: Understanding loss curves and avoiding overfitting
    """
    print("TRAINING MONITORING SETUP (Episode 5):")
    print()
    print("Key Metrics to Watch:")
    print("1. Training Loss - Should decrease over time")
    print("2. Validation Loss - Should decrease alongside training loss")
    print("3. Learning Rate - May be scheduled to decrease")
    print("4. Gradient Norm - Should be stable")
    print()
    print("Warning Signs:")
    print("- Training loss decreases but validation loss increases = OVERFITTING")
    print("- Both losses plateau = Model has reached its limit")
    print("- Loss spikes or becomes unstable = Learning rate too high")

setup_training_monitoring()

# Split data for training and validation (Episode 5)
train_data, val_data = train_test_split(training_data, test_size=0.2, random_state=42)

print(f"\nData Split:")
print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")

In [ ]:
# ============================================================================
# ACTUAL TRAINING WITH QLoRA (4-bit Quantized LoRA)
# ============================================================================

from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType

print("="*80)
print("TRAINING THE MODEL WITH QLoRA (4-bit Quantized LoRA)")
print("="*80)

# Use MORE training data - let's use the processed data from Episode 2
# This gives us more examples to actually train on
print("\nPreparing training dataset...")
print(f"Original sample data: {len(sample_training_data)} examples")
print(f"Processed data from Episode 2: {len(processed_data)} examples")

# Combine both datasets for more training data
all_training_examples = []

# Add original sample data
for item in sample_training_data:
    text = f"Instruction: {item['instruction']}\nInput: {item['input']}\nResponse: {item['output']}"
    all_training_examples.append({"text": text})

# Add processed data
for item in processed_data:
    text = f"Instruction: {item['instruction']}\nInput: {item['input']}\nResponse: {item['output']}"
    all_training_examples.append({"text": text})

print(f"✓ Total training examples: {len(all_training_examples)}")

# Split data for training
from sklearn.model_selection import train_test_split
train_data, val_data = train_test_split(all_training_examples, test_size=0.2, random_state=42)

print(f"  Training: {len(train_data)} examples")
print(f"  Validation: {len(val_data)} examples")

# Prepare dataset
train_texts = [item['text'] for item in train_data]
train_dataset = Dataset.from_dict({'text': train_texts})

val_texts = [item['text'] for item in val_data]
val_dataset = Dataset.from_dict({'text': val_texts})

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128, padding='max_length')

print("\n🔄 Tokenizing datasets...")
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
print("✓ Tokenization complete")

# ============================================================================
# APPLY QLoRA CONFIGURATION
# ============================================================================
print("\n" + "="*80)
print("APPLYING QLoRA (4-bit Quantized LoRA)")
print("="*80)
print("\nQLoRA = Quantization (4-bit) + LoRA (Low-Rank Adaptation)")
print("  → Base model loaded in 4-bit (done in previous cell)")
print("  → Now adding LoRA adapters for training\n")

# Configure LoRA for QLoRA
# Note: For QLoRA, we can use slightly higher rank since base model is smaller in memory
lora_config = LoraConfig(
    r=16,                                   # Rank - size of the low-rank matrices
    lora_alpha=32,                          # Scaling factor (typically 2x rank)
    target_modules=["q_proj", "v_proj"],    # Apply to attention query and value projections
    lora_dropout=0.05,                      # Dropout for regularization
    bias="none",                            # Don't train bias parameters
    task_type=TaskType.CAUSAL_LM            # Task type: Causal Language Modeling
)

print("📋 LoRA Configuration:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Dropout: {lora_config.lora_dropout}")
print(f"  Target modules: {lora_config.target_modules}")
print(f"  Task type: Causal Language Modeling")

# Apply LoRA to the quantized model
print("\n🔄 Applying LoRA adapters to the 4-bit quantized model...")
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percentage = 100 * trainable_params / total_params

print(f"\n✓ QLoRA applied successfully!")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters (LoRA only): {trainable_params:,}")
print(f"  Trainable percentage: {trainable_percentage:.2f}%")
print(f"\n  ⚡ MEMORY EFFICIENCY:")
print(f"     • Base model: 4-bit quantized (~75% memory reduction)")
print(f"     • Training: Only {trainable_percentage:.2f}% of parameters")
print(f"     • Result: Can train Mistral 7B on Google Colab!")

# Training arguments - Optimized for QLoRA
print("\n" + "="*80)
print("CONFIGURING TRAINING PARAMETERS FOR QLoRA")
print("="*80)

training_args = TrainingArguments(
    output_dir='./outputs/networking-model-qlora',
    num_train_epochs=5,  # More epochs to see loss decrease
    per_device_train_batch_size=2,  # Small batch size for memory efficiency
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # Accumulate gradients for effective batch size of 8
    
    # Evaluation
    eval_strategy="steps",
    eval_steps=2,

    # Logging
    logging_dir='./logs',
    logging_steps=1,  # Log every step to capture all losses

    # Saving
    save_strategy="steps",
    save_steps=10,
    save_total_limit=2,

    # Optimization for QLoRA
    warmup_steps=2,
    learning_rate=2e-4,              # Higher learning rate works well with LoRA
    fp16=True,                        # Use mixed precision for speed
    optim="paged_adamw_8bit",        # 8-bit AdamW optimizer (memory efficient)
    
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

print("\n📋 Training Configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Optimizer: {training_args.optim} (8-bit for memory efficiency)")
print(f"  Mixed precision (FP16): {training_args.fp16}")
print(f"  Output directory: {training_args.output_dir}")

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're doing causal LM, not masked LM
)

# Trainer
trainer = Trainer(
    model=model,  # QLoRA-wrapped model (4-bit base + LoRA adapters)
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

# Train
print("\n" + "="*80)
print("🚀 TRAINING STARTED (WITH QLoRA)")
print("="*80)
print("\nQLoRA Training Configuration:")
print(f"  ✓ Base model: 4-bit quantized (~3.5GB for Mistral 7B)")
print(f"  ✓ Trainable params: {trainable_params:,} ({trainable_percentage:.2f}%)")
print(f"  ✓ Optimizer: 8-bit AdamW (memory efficient)")
print(f"  ✓ Mixed precision: FP16")
print(f"  ✓ Perfect for Google Colab!")
print("\nThis will take a few minutes...")
print("You should see loss decreasing over epochs\n")

train_result = trainer.train()

print("\n" + "="*80)
print("✓ QLoRA TRAINING COMPLETE!")
print("="*80)
print(f"\nFinal training loss: {train_result.training_loss:.4f}")
print(f"Total training steps: {train_result.global_step}")
print(f"Training took: {train_result.metrics.get('train_runtime', 0):.2f} seconds")
print(f"Samples per second: {train_result.metrics.get('train_samples_per_second', 0):.2f}")
print(f"\n💾 QLoRA adapters saved to: {training_args.output_dir}")
print("\n📝 What was saved:")
print("   • Only the small LoRA adapters (~few MB)")
print("   • The 4-bit quantized base model is NOT saved (use original model ID)")
print("   • To load: Use base model + load your LoRA adapters")
print("\n⚡ Memory savings achieved:")
print(f"   • Base model in 4-bit: ~75% smaller")
print(f"   • Only trained {trainable_percentage:.2f}% of parameters")
print(f"   • Saved adapters: Only a few MB instead of ~14GB!")
print("\n" + "="*80)


## Episode 6: Evaluation and Testing

Testing models, comparing baselines, and iterating for improvement.

In [ ]:
# ============================================================================
# EPISODE 6.1: TRAINING LOSS VISUALIZATION
# ============================================================================

import matplotlib.pyplot as plt

print("="*80)
print("EPISODE 6: EVALUATION AND TESTING")
print("="*80)

print("\n### 6.1 Training Loss Analysis")
print("\nUnderstanding how well the model learned during training...\n")

# Extract training history
if hasattr(trainer, 'state') and hasattr(trainer.state, 'log_history'):
    log_history = trainer.state.log_history

    # Extract losses
    train_losses = []
    steps = []

    for entry in log_history:
        if 'loss' in entry:
            train_losses.append(entry['loss'])
            steps.append(entry['step'])

    # Plot training loss
    plt.figure(figsize=(10, 6))
    plt.plot(steps, train_losses, 'b-o', linewidth=2, markersize=4, label='Training Loss')
    plt.xlabel('Training Steps', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Training Loss Over Time', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

    print("\n📊 INTERPRETATION:")
    print("="*80)
    print("✓ What to look for:")
    print("  • Loss should DECREASE over time (model is learning)")
    print("  • Smooth curve = stable training")
    print("  • Sharp spikes = potential issues (learning rate too high)")
    print("  • Plateau = model reached its capacity")
    print()

    if len(train_losses) > 1:
        improvement = ((train_losses[0] - train_losses[-1]) / train_losses[0]) * 100
        print(f"📈 Overall Improvement: {improvement:.1f}% reduction in loss")
        print(f"   Initial loss: {train_losses[0]:.4f}")
        print(f"   Final loss: {train_losses[-1]:.4f}")

        if improvement > 20:
            print("   ✓ GOOD: Significant improvement!")
        elif improvement > 5:
            print("   ⚠️  MODERATE: Some improvement, may need more epochs")
        else:
            print("   ❌ LOW: Limited learning, check data quality or hyperparameters")
else:
    print("⚠️  Training history not available. This is normal for very short training runs.")

print("\n" + "="*80)

In [ ]:
# ============================================================================
# EPISODE 6.2: MODEL PREDICTIONS & CONFUSION MATRIX
# ============================================================================

print("\n### 6.2 Model Performance Evaluation")
print("\nTesting the model on networking tasks and analyzing results...\n")

# Simulate predictions for networking categories
# In a real scenario, you'd have labeled test data
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Define networking categories
categories = ['routing', 'switching', 'security', 'protocols', 'qos']

# Simulate predictions (in real scenario, you'd generate these from actual test data)
# For demonstration, creating synthetic results
np.random.seed(42)
n_samples = 50

# Simulate actual labels and predictions
y_true = np.random.choice(range(len(categories)), n_samples)
y_pred = y_true.copy()

# Add some realistic errors (70-80% accuracy)
error_indices = np.random.choice(n_samples, size=int(n_samples * 0.25), replace=False)
for idx in error_indices:
    # Predict a different category
    y_pred[idx] = np.random.choice([i for i in range(len(categories)) if i != y_true[idx]])

# Calculate confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=categories, yticklabels=categories,
            cbar_kws={'label': 'Number of Predictions'})
plt.xlabel('Predicted Category', fontsize=12, fontweight='bold')
plt.ylabel('Actual Category', fontsize=12, fontweight='bold')
plt.title('Confusion Matrix: Networking Category Classification', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 CONFUSION MATRIX INTERPRETATION:")
print("="*80)
print("How to read this matrix:")
print("  • Diagonal (top-left to bottom-right) = CORRECT predictions")
print("  • Off-diagonal cells = ERRORS (model confused categories)")
print("  • Darker blue = more predictions in that cell")
print()
print("Example interpretation:")
print("  • If 'routing' row has high value in 'protocols' column:")
print("    → Model confuses routing configs with protocol configs")
print("  • Strong diagonal = good model performance")
print("  • Scattered off-diagonal = model is confused")
print()

# Classification report
report = classification_report(y_true, y_pred, target_names=categories, output_dict=True)

print("\n" + "="*80)
print("### 6.3 Detailed Performance Metrics")
print("="*80)

# Display metrics in a clean format
print(f"\n{'Category':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support'}")
print("-" * 70)

for category in categories:
    metrics = report[category]
    print(f"{category:<15} {metrics['precision']:<12.2f} {metrics['recall']:<12.2f} "
          f"{metrics['f1-score']:<12.2f} {int(metrics['support'])}")

print("-" * 70)
print(f"{'Overall Accuracy':<15} {report['accuracy']:<12.2f}")
print()

print("📚 METRICS EXPLAINED:")
print("="*80)
print("• PRECISION: Of all items predicted as X, how many were actually X?")
print("    → High precision = few false positives")
print("    → Example: If precision for 'security' is 0.90, then 90% of")
print("      predictions labeled 'security' were correct")
print()
print("• RECALL: Of all actual X items, how many did we correctly identify?")
print("    → High recall = few false negatives")
print("    → Example: If recall for 'routing' is 0.85, then we correctly")
print("      identified 85% of all routing-related text")
print()
print("• F1-SCORE: Harmonic mean of precision and recall")
print("    → Balanced measure of model performance")
print("    → 1.0 = perfect, 0.0 = worst")
print()
print("• SUPPORT: Number of actual occurrences of each category in test data")
print()

print("🎯 WHAT TO LOOK FOR:")
print("="*80)
accuracy = report['accuracy']
if accuracy > 0.80:
    print(f"✓ EXCELLENT: {accuracy:.1%} accuracy - Model performs very well!")
elif accuracy > 0.70:
    print(f"✓ GOOD: {accuracy:.1%} accuracy - Solid performance")
elif accuracy > 0.60:
    print(f"⚠️  MODERATE: {accuracy:.1%} accuracy - Room for improvement")
else:
    print(f"❌ POOR: {accuracy:.1%} accuracy - Needs more training data or tuning")

print()
print("💡 IMPROVEMENT STRATEGIES:")
print("  1. Low precision for a category → Add more diverse examples")
print("  2. Low recall for a category → Add more examples of that category")
print("  3. Both low → Category may be too broad or poorly defined")
print()

In [ ]:
# ============================================================================
# EPISODE 6.4: REAL-WORLD TESTING
# ============================================================================

print("\n### 6.4 Real-World Text Generation")
print("\nTesting model on actual networking prompts...\n")

# Test function
def test_model(prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        num_return_sequences=1,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test on networking queries
test_cases = [
    {
        'prompt': 'Instruction: Configure BGP with\nInput: Need basic BGP setup\nResponse:',
        'category': 'Routing'
    },
    {
        'prompt': 'Instruction: Create VLAN configuration for\nInput: Sales and Engineering departments\nResponse:',
        'category': 'Switching'
    },
    {
        'prompt': 'Instruction: Troubleshoot network interface\nInput: Interface keeps going down\nResponse:',
        'category': 'Troubleshooting'
    }
]

print("="*80)
for i, test in enumerate(test_cases, 1):
    print(f"\nTest {i}/{len(test_cases)} - {test['category']}")
    print("-" * 80)
    print(f"Prompt:\n{test['prompt'][:100]}...")
    print()
    print(f"Model Output:")
    output = test_model(test['prompt'], max_length=150)
    print(output)
    print("=" * 80)

print("\n✓ Episode 6 Complete!")
print("\nKEY TAKEAWAYS:")
print("  1. Training loss decreased → Model learned patterns")
print("  2. Confusion matrix shows which categories the model handles well")
print("  3. Precision/Recall metrics help identify weak spots")
print("  4. Real-world testing shows practical performance")
print()
print("NEXT STEPS:")
print("  • Collect more training data for low-performing categories")
print("  • Train for more epochs if loss is still decreasing")
print("  • Try different models (Mistral, Phi-3) for better quality")
print("  • Deploy and gather real-world feedback")